# AMASS 데이터 살펴보기

위치: `HumanML3D/amass_data`. 모션 1개가 `.npz` 파일 1개로 저장되어 있다.

## 1. 데이터셋 종류

`amass_data` 아래 폴더 하나가 데이터셋 하나다. `shape.npz`는 모션이 아니어서 모션 파일 수에서 뺀다.

In [1]:
# 폴더를 다루는 라이브러리
import os
# .npz 파일을 읽는 라이브러리
import numpy as np
# 표를 만드는 라이브러리
import pandas as pd

# AMASS 폴더 경로
root = '/home/jaesik/mnt/HumanML3D/amass_data'
# 압축 파일을 빼고 폴더만 골라 데이터셋 이름 목록 만들기
datasets = sorted(d for d in os.listdir(root) if os.path.isdir(f'{root}/{d}'))

# 모든 .npz 파일의 경로를 데이터셋별로 모으기
files = {d: [os.path.join(folder, f) for folder, _, names in os.walk(f'{root}/{d}') for f in names if f.endswith('.npz')] for d in datasets}

# 표의 행을 담을 목록
rows = []
# 데이터셋을 하나씩 꺼내기
for d in datasets:
    # shape.npz를 뺀 모션 파일만 고르기
    motions = [f for f in files[d] if os.path.basename(f) != 'shape.npz']
    # 파일별 프레임 속도와 재생 시간(초)을 담을 목록
    fps, seconds = [], []
    # 모션 파일을 하나씩 열기
    for f in motions:
        # 파일 열기
        data = np.load(f, allow_pickle=True)
        # 프레임 속도를 소수 첫째 자리까지만 남기기
        fps.append(round(float(data['mocap_framerate']), 1))
        # 프레임 수를 프레임 속도로 나눠 재생 시간 구하기
        seconds.append(len(data['trans']) / fps[-1])
    # 이 데이터셋의 요약을 표에 추가
    rows.append({'dataset': d, 'subfolders': sum(os.path.isdir(f'{root}/{d}/{x}') for x in os.listdir(f'{root}/{d}')), 'motion files': len(motions), 'fps': sorted(set(fps)), 'hours': round(sum(seconds) / 3600, 2)})

# 표 만들기
table = pd.DataFrame(rows)
# 전체 데이터셋 수, 모션 파일 수, 시간 출력
print(len(table), 'datasets,', table['motion files'].sum(), 'motion files,', round(table['hours'].sum(), 1), 'hours')
# 표 출력
table

18 datasets, 13943 motion files, 45.2 hours


,dataset,subfolders,motion files,fps,hours
0,ACCAD,20,252,[120.0],0.45
1,BMLhandball,10,649,[120.0],1.70
2,BMLmovi,86,1801,[120.0],2.82
3,BioMotionLab_NTroje,111,3061,[120.0],8.71
4,CMU,102,2082,"[60.0, 120.0]",9.19
5,DFaust_67,10,129,[60.0],0.19
6,EKUT,4,349,[100.0],0.51
7,Eyes_Japan_Dataset,12,750,"[120.0, 250.0]",6.06
8,HumanEva,3,28,[120.0],0.14
9,KIT,55,4232,[100.0],11.03


## 2. 파일 종류

파일마다 들어 있는 키 조합을 세면 3가지다.

In [2]:
# 키 조합별로 파일 수를 세어 담을 목록
counts = []
# 데이터셋을 하나씩 꺼내기
for d in datasets:
    # 파일을 하나씩 꺼내기
    for f in files[d]:
        # 파일 안의 키 이름을 정렬해 하나의 글자로 합치기
        counts.append({'dataset': d, 'keys': ', '.join(sorted(np.load(f, allow_pickle=True).files))})

# 표의 긴 글자를 자르지 않고 모두 표시
pd.set_option('display.max_colwidth', None)
# 키 조합별 파일 수와 해당 데이터셋 목록 출력
pd.DataFrame(counts).groupby('keys')['dataset'].agg(files='count', datasets=lambda s: ', '.join(sorted(set(s))))

,files,datasets
keys,,
"betas, dmpls, gender, marker_data, marker_labels, mocap_framerate, poses, trans",269,"DFaust_67, SSM_synced, Transitions_mocap"
"betas, dmpls, gender, mocap_framerate, poses, trans",13674,"ACCAD, BMLhandball, BMLmovi, BioMotionLab_NTroje, CMU, EKUT, Eyes_Japan_Dataset, HumanEva, KIT, MPI_HDM05, MPI_Limits, MPI_mosh, SFU, TCD_handMocap, TotalCapture"
"betas, gender",112,"BMLhandball, BMLmovi, CMU, DFaust_67"


## 3. 파일 안의 값과 형태

모션 파일 하나를 예시로 연다. `T`는 프레임 수이고 파일마다 다르다.

In [3]:
# 예시 모션 파일 열기
amass = np.load(f'{root}/KIT/3/kick_high_left02_poses.npz', allow_pickle=True)
# 키를 하나씩 꺼내기
for key in amass.files:
    # 키 이름, 배열 크기, 자료형 출력
    print(f'{key:16}{str(amass[key].shape):12}{amass[key].dtype}')

trans           (582, 3)    float64
gender          ()          <U4
mocap_framerate ()          float64
betas           (16,)       float64
dmpls           (582, 8)    float64
poses           (582, 156)  float64


| 키 | 형태 | 내용 |
|---|---|---|
| `poses` | (T, 156) | 프레임별 관절 회전. 축-각 표현, 라디안 |
| `trans` | (T, 3) | 프레임별 몸 전체의 위치. 미터, z축이 위쪽 |
| `betas` | (16,) | 체형 값. 프레임과 무관하게 파일당 1벌 |
| `dmpls` | (T, 8) | 프레임별 연조직(살) 움직임 값. 위 예시 파일은 전부 0 |
| `gender` | 값 1개 | 성별. 체형 모델을 고를 때 사용. `SSM_synced`는 글자가 아닌 bytes로 저장됨 |
| `mocap_framerate` | 값 1개 | 촬영 프레임 속도 (fps) |
| `marker_data` | (T, M, 3) | 모션 캡처 마커 위치. 일부 파일만 있음. M은 마커 수 |
| `marker_labels` | (M,) | 마커 이름. 일부 파일만 있음 |

`shape.npz`에는 `betas`와 `gender`만 있다.

`poses` 156개는 관절 52개 × 회전값 3개다. 한 관절이 연속된 3칸을 차지한다.

| 번호 | 관절 수 | 내용 |
|---|---|---|
| 0–2 | 1 | 몸 전체의 방향 (pelvis) |
| 3–65 | 21 | 몸 관절 |
| 66–155 | 30 | 손가락 관절 (한 손 15개) |